# SMAE Engine: Prototyping Sandbox Verification

Verifies the `IngestionPipeline` extraction logic end-to-end.

- **Sections 1–4**: pure Python, no GCP credentials required.
- **Section 5**: live DoD gate — requires ADC and an uploaded PDF.

In [ ]:
import io
import sys
sys.path.append("../..")

from datetime import datetime, timezone
from unittest.mock import patch, MagicMock

from pypdf import PdfReader, PdfWriter

from backend.smae_engine.source_code.config import SmaeSettings
from backend.smae_engine.source_code.main import IngestionPipeline
from backend.smae_engine.source_code.schemas import (
    ExtractionMetadata,
    ExtractionRequest,
    ExtractionResponse,
    ExtractResponse,
    FoodItem,
    VerificationResponse,
)

print("Imports OK")

## Section 1 — Schema Validation (No GCP Required)

Verify Pydantic models enforce all constraints: URI format, `max_length`, `ge=0`, and `VerificationResponse`.

In [ ]:
from pydantic import ValidationError

# Valid URI
req = ExtractionRequest(gcs_uri="gs://nutritional-data-sources/smae.pdf")
print(f"Valid URI accepted: {req.gcs_uri}")

# Invalid scheme rejected
try:
    ExtractionRequest(gcs_uri="https://evil.com/file.pdf")
except ValidationError:
    print("Invalid scheme rejected: ValidationError")

# Untrusted bucket rejected by _validate_trusted_bucket (not schema) — shown in Section 3

# FoodItem max_length enforced
try:
    FoodItem(food_uuid="x", food="a" * 201, ingested_at=datetime.now(timezone.utc))
except ValidationError:
    print("food max_length=200 enforced: ValidationError")

# Negative nutrient rejected
try:
    FoodItem(food_uuid="x", energy_kcal=-1, ingested_at=datetime.now(timezone.utc))
except ValidationError:
    print("Negative energy_kcal rejected: ValidationError")

# VerificationResponse
vr = VerificationResponse(status="valid", items_count=10)
print(f"VerificationResponse: status={vr.status!r}, items_count={vr.items_count}")
print("Section 1 passed")

## Section 2 — UUID Determinism (No GCP Required)

Confirm `_generate_food_uuid` produces stable, collision-free identifiers.

In [ ]:
pipeline = IngestionPipeline()

uuid_apple_1 = pipeline._generate_food_uuid("gs://nutritional-data-sources/smae.pdf", "Apple")
uuid_apple_2 = pipeline._generate_food_uuid("gs://nutritional-data-sources/smae.pdf", "Apple")
uuid_banana  = pipeline._generate_food_uuid("gs://nutritional-data-sources/smae.pdf", "Banana")
uuid_other   = pipeline._generate_food_uuid("gs://other-bucket/smae.pdf", "Apple")

assert uuid_apple_1 == uuid_apple_2, "UUID must be deterministic"
assert uuid_apple_1 != uuid_banana,  "Different food names must yield different UUIDs"
assert uuid_apple_1 != uuid_other,   "Different source URIs must yield different UUIDs"

print(f"Apple UUID : {uuid_apple_1}")
print(f"Banana UUID: {uuid_banana}")
print("Section 2 passed")

## Section 3 — Transform + Verify In-Memory (No GCP Required)

Simulate the pipeline using synthetic data to validate `transform` and `verify` logic, including edge cases.

In [ ]:
SOURCE_URI = "gs://nutritional-data-sources/smae.pdf"

raw_items = [
    {"food": "Apple",          "food_group": "FRUITS", "energy_kcal": 52,  "protein_g": 0.3, "lipids_g": 0.2, "carbohydrates_g": 14.0},
    {"food": "Chicken breast",  "food_group": "MEAT",   "energy_kcal": 165, "protein_g": 31.0, "lipids_g": 3.6, "carbohydrates_g": 0.0},
    {"food": None,              "food_group": "OTHER",  "energy_kcal": 0},
]

extract_result = ExtractResponse(raw_items=raw_items, source_uri=SOURCE_URI)
transform_result = pipeline.transform(extract_result)

print(f"Transformed {len(transform_result.items)} items:")
for item in transform_result.items:
    print(f"  {item.food or '[null food]'!r:<20} uuid={item.food_uuid[:8]}...  kcal={item.energy_kcal}")

# Verify
metadata = ExtractionMetadata(source_uri=SOURCE_URI, processed_at=datetime.now(timezone.utc))
response = ExtractionResponse(items=transform_result.items, metadata=metadata)
vr = pipeline.verify(response)

assert vr.status == "valid"
assert vr.items_count == 3
print(f"\nVerification: status={vr.status!r}, items_count={vr.items_count}")

# Edge case: trusted bucket validation
try:
    pipeline._validate_trusted_bucket("gs://attacker-bucket/evil.pdf")
except ValueError as e:
    print(f"Untrusted bucket rejected: {type(e).__name__}")

print("Section 3 passed")

## Section 4 — Parallel Batch In-Memory Simulation (No GCP Required)

Verifies the parallel batch logic using a synthetic in-memory PDF:
- `_resolve_pages`: all pages, `page_range`, specific `pages`, clamping, out-of-range rejection
- `_build_batches`: even split, remainder, single-page batches
- `_split_pdf_pages`: page count in extracted PDF
- `run()` routing: `page_range` and `pages` both trigger `extract_parallel`
- `_process_batches_parallel`: order-preserving merge with mocked Gemini calls

In [ ]:
import io
from pypdf import PdfReader, PdfWriter
from unittest.mock import patch, MagicMock
from backend.smae_engine.source_code.schemas import ExtractionRequest

def make_pdf(num_pages: int) -> bytes:
    writer = PdfWriter()
    for _ in range(num_pages):
        writer.add_blank_page(width=100, height=100)
    buf = io.BytesIO()
    writer.write(buf)
    return buf.getvalue()

pdf_5 = make_pdf(5)
parallel_pipeline = IngestionPipeline()

# --- _resolve_pages ---
req_all = ExtractionRequest(gcs_uri="gs://nutritional-data-sources/smae.pdf")
assert parallel_pipeline._resolve_pages(req_all, pdf_5) == [1, 2, 3, 4, 5]
print("resolve_pages all pages:     OK")

req_range = ExtractionRequest(gcs_uri="gs://nutritional-data-sources/smae.pdf", page_range=(2, 4))
assert parallel_pipeline._resolve_pages(req_range, pdf_5) == [2, 3, 4]
print("resolve_pages page_range:    OK")

req_specific = ExtractionRequest(gcs_uri="gs://nutritional-data-sources/smae.pdf", pages=[1, 3, 5])
assert parallel_pipeline._resolve_pages(req_specific, pdf_5) == [1, 3, 5]
print("resolve_pages specific pages: OK")

req_clamp = ExtractionRequest(gcs_uri="gs://nutritional-data-sources/smae.pdf", page_range=(3, 100))
assert parallel_pipeline._resolve_pages(req_clamp, pdf_5) == [3, 4, 5]
print("resolve_pages clamps range:   OK")

try:
    req_oor = ExtractionRequest(gcs_uri="gs://nutritional-data-sources/smae.pdf", pages=[2, 99])
    parallel_pipeline._resolve_pages(req_oor, pdf_5)
    print("ERROR: out-of-range pages should have raised")
except ValueError as e:
    print(f"resolve_pages out-of-range rejected: {type(e).__name__}")

# --- _build_batches ---
assert parallel_pipeline._build_batches([1, 2, 3, 4, 5], 2) == [[1, 2], [3, 4], [5]]
assert parallel_pipeline._build_batches([1, 2, 3], 5) == [[1, 2, 3]]
assert parallel_pipeline._build_batches([1, 2, 3], 1) == [[1], [2], [3]]
print("build_batches:               OK")

# --- _split_pdf_pages ---
pdf_3 = parallel_pipeline._split_pdf_pages(pdf_5, [1, 3, 5])
assert len(PdfReader(io.BytesIO(pdf_3)).pages) == 3
print("split_pdf_pages:             OK")

# --- _process_batches_parallel (mocked Gemini) ---
def fake_extract_batch(pdf_bytes, pages):
    return [{"food": f"item-p{p}"} for p in pages]

with patch.object(parallel_pipeline, "_extract_single_batch", side_effect=fake_extract_batch):
    result = parallel_pipeline._process_batches_parallel(pdf_5, [[1, 2], [3, 4], [5]])

assert result == [
    {"food": "item-p1"}, {"food": "item-p2"},
    {"food": "item-p3"}, {"food": "item-p4"},
    {"food": "item-p5"},
], f"Unexpected result: {result}"
print("process_batches_parallel:    OK (order preserved)")

# --- run() routing (page_range → extract_parallel) ---
from backend.smae_engine.source_code.schemas import ExtractResponse, TransformResponse, VerificationResponse
from backend.smae_engine.source_code.config import SmaeSettings

routing_pipeline = IngestionPipeline(settings=SmaeSettings(trusted_bucket="nutritional-data-sources"))
req_with_range = ExtractionRequest(
    gcs_uri="gs://nutritional-data-sources/smae.pdf",
    page_range=(1, 5),
)
with patch.object(routing_pipeline, "extract_parallel",
                  return_value=ExtractResponse(raw_items=[], source_uri=req_with_range.gcs_uri)) as mock_ep, \
     patch.object(routing_pipeline, "extract") as mock_e, \
     patch.object(routing_pipeline, "transform",
                  return_value=TransformResponse(items=[])) as mock_t, \
     patch.object(routing_pipeline, "verify",
                  return_value=VerificationResponse(status="valid", items_count=0)):
    routing_pipeline.run(req_with_range)
    assert mock_ep.called and not mock_e.called
print("run() routes page_range → extract_parallel: OK")

req_with_pages = ExtractionRequest(
    gcs_uri="gs://nutritional-data-sources/smae.pdf",
    pages=[1, 2, 3],
)
with patch.object(routing_pipeline, "extract_parallel",
                  return_value=ExtractResponse(raw_items=[], source_uri=req_with_pages.gcs_uri)) as mock_ep2, \
     patch.object(routing_pipeline, "extract") as mock_e2, \
     patch.object(routing_pipeline, "transform",
                  return_value=TransformResponse(items=[])) as mock_t2, \
     patch.object(routing_pipeline, "verify",
                  return_value=VerificationResponse(status="valid", items_count=0)):
    routing_pipeline.run(req_with_pages)
    assert mock_ep2.called and not mock_e2.called
print("run() routes pages list → extract_parallel: OK")

print("\nSection 4 passed")

## Section 5 — Live Parallel Extraction (DoD Gate — Requires GCP)

**Prerequisites before running:**
1. `gcloud auth application-default login`
2. `./backend/smae_engine/source_code/create_resources.sh`
3. Upload a SMAE PDF: `gcloud storage cp <local_smae.pdf> gs://nutritional-data-sources/sample.pdf`

This cell must execute with 100% extraction success to satisfy the Stage 1 DoD.

The `IngestionPipeline.run()` method automatically routes to:
- `extract()` — single-pass, full-document extraction when no page params are set.
- `extract_parallel()` — parallel batch extraction when `page_range` or `pages` is provided.

Use the parallel mode for large PDFs (e.g., the full SMAE catalog) to reduce latency.

In [ ]:
# Uncomment after completing all prerequisites above.

# Full document (single-pass):
# pipeline_live = IngestionPipeline()
# request = ExtractionRequest(gcs_uri="gs://nutritional-data-sources/smae.pdf")
# response = pipeline_live.run(request)

# Parallel batch (pages 1–50, 5 pages per batch):
# pipeline_live = IngestionPipeline()
# request = ExtractionRequest(
#     gcs_uri="gs://nutritional-data-sources/smae.pdf",
#     page_range=(1, 50),
# )
# response = pipeline_live.run(request)

# Specific pages:
pipeline_live = IngestionPipeline()
request = ExtractionRequest(
    gcs_uri="gs://nutritional-data-sources/SMAE-14-54.pdf",
    # pages=[14,15],
)
response = pipeline_live.run(request)

# print(f"Extracted {len(response.items)} items")
# print(f"Source   : {response.metadata.source_uri}")
# print(f"Processed: {response.metadata.processed_at}")
# print()
# for item in response.items[:5]:
#     print(item.model_dump_json(indent=2))

2026-04-26 00:54:12.297 | INFO     | backend.smae_engine.source_code.main:run:53 - Pipeline started for bucket: nutritional-data-sources
2026-04-26 00:54:12.298 | INFO     | backend.smae_engine.source_code.main:extract_parallel:97 - Starting parallel extraction from bucket: nutritional-data-sources
2026-04-26 00:54:13.470 | DEBUG    | backend.smae_engine.source_code.main:_download_pdf:219 - Downloading 23.6 MB from bucket: nutritional-data-sources
Ignoring wrong pointing object 9 0 (offset 0)
Ignoring wrong pointing object 11 0 (offset 0)
Ignoring wrong pointing object 26 0 (offset 0)
Ignoring wrong pointing object 159 0 (offset 0)
Ignoring wrong pointing object 192 0 (offset 0)
Ignoring wrong pointing object 275 0 (offset 0)
Ignoring wrong pointing object 439 0 (offset 0)
Ignoring wrong pointing object 441 0 (offset 0)
Ignoring wrong pointing object 443 0 (offset 0)
Ignoring wrong pointing object 536 0 (offset 0)
Ignoring wrong pointing object 538 0 (offset 0)
Ignoring wrong pointing 

In [7]:
len(response.items)

52

In [14]:
response.items[6].ingested_at.date()

datetime.date(2026, 4, 26)